# UKRI FoR Classifier — POC Batch Inference
## Single Joblib Model Release

POC flow:
1. Fetch input from S3 with boto3
2. Validate stakeholder-agreed columns
3. Use `ApplicationID + ApplicationOriginSource` as application identity
4. Exclude rows where both title and summary are empty
5. Reuse `data_preparation.py`
6. Load a single `.joblib` containing `vectorizer`, `models`, `thresholds`, `mlb`
7. Run multi-label inference
8. Identify unresolved primary predictions
9. Leave a placeholder for the future fallback model
10. Build, validate, save and upload the final output

## 0. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import json, os, sys

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "models").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
REJECTED_DIR = DATA_DIR / "rejected"
UNRESOLVED_DIR = DATA_DIR / "unresolved"
LOG_DIR = PROJECT_ROOT / "logs"
DATA_PREPARATION_DIR = PROJECT_ROOT

for d in [INPUT_DIR, OUTPUT_DIR, REJECTED_DIR, UNRESOLVED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET = "<YOUR-STAGING-BUCKET>"
S3_INPUT_KEY = None
S3_INPUT_PREFIX = "<YOUR-INPUT-PREFIX>/"
S3_OUTPUT_PREFIX = "<YOUR-OUTPUT-PREFIX>/"
INPUT_SUFFIXES = (".csv", ".parquet")

MAIN_MODEL_PATH = PROJECT_ROOT / "models" / "<PRIMARY_MODEL>.joblib"

MODEL_TEXT_FIELDS = ["ApplicationTitle", "ApplicationSummary"]
TAXONOMY_FILE_TOKEN = "FoR"
TAXONOMY_VALUE = "FieldsOfResearch"
MODEL_NAME = "FoRClassification"
MODEL_VERSION = "1.1"
SCORE_TYPE = "uncalibrated"
OUTPUT_EXTENSION = ".csv"
N_JOBS = 4

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAIN_MODEL_PATH:", MAIN_MODEL_PATH)

## 1. Imports and environment checks

In [ ]:
import boto3, botocore, joblib, numpy as np, pandas as pd, sklearn, spacy

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("spaCy:", spacy.__version__)
print("joblib:", joblib.__version__)

if not MAIN_MODEL_PATH.exists():
    raise FileNotFoundError(f"Primary model artefact not found: {MAIN_MODEL_PATH}")

## 2. Import shared preprocessing

`data_preparation.py` should expose `preprocess_text_fields`.

For production, remove/disable any runtime `spacy.cli.download("en_core_web_sm")`; package the approved spaCy model with the environment/container.

In [ ]:
if str(DATA_PREPARATION_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARATION_DIR))

from data_preparation import preprocess_text_fields
print("Imported preprocess_text_fields successfully.")

## 3. Test AWS identity and S3 access

In [ ]:
session = boto3.Session()
s3 = session.client("s3")
sts = session.client("sts")

identity = sts.get_caller_identity()
print("AWS account:", identity.get("Account"))
print("Caller ARN:", identity.get("Arn"))

s3.head_bucket(Bucket=S3_BUCKET)
print(f"S3 access OK: s3://{S3_BUCKET}")

## 4. Locate the input object

In [ ]:
def find_latest_s3_object(s3_client, bucket, prefix, allowed_suffixes=(".csv",".parquet")):
    paginator = s3_client.get_paginator("list_objects_v2")
    candidates = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].lower().endswith(tuple(s.lower() for s in allowed_suffixes)):
                candidates.append(obj)
    if not candidates:
        raise FileNotFoundError(f"No matching files under s3://{bucket}/{prefix}")
    return max(candidates, key=lambda x: x["LastModified"])["Key"]

selected_input_key = S3_INPUT_KEY or find_latest_s3_object(
    s3, S3_BUCKET, S3_INPUT_PREFIX, INPUT_SUFFIXES
)
print(f"s3://{S3_BUCKET}/{selected_input_key}")

## 5. Download input to Ronin

In [ ]:
local_input_path = INPUT_DIR / Path(selected_input_key).name
s3.download_file(S3_BUCKET, selected_input_key, str(local_input_path))

if not local_input_path.exists() or local_input_path.stat().st_size == 0:
    raise ValueError("Input download failed or file is empty.")

print("Downloaded:", local_input_path)

## 6. Read input

In [ ]:
def read_input_file(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported format: {path.suffix}")

df_raw = read_input_file(local_input_path)
print("Rows:", len(df_raw))
display(df_raw.head(3))

## 7. Validate input contract

In [ ]:
REQUIRED_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "ApplicationTitle",
    "ApplicationSummary",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if len(df_raw) == 0:
    raise ValueError("Input contains zero rows.")

df = df_raw.copy()
df["ApplicationID"] = df["ApplicationID"].astype("string")
df["ApplicationOriginSource"] = df["ApplicationOriginSource"].astype("string")

if df["ApplicationOriginSource"].isna().any():
    raise ValueError("ApplicationOriginSource contains null values.")

df["_application_key"] = (
    df["ApplicationID"].fillna("<NULL>") + "||" +
    df["ApplicationOriginSource"].fillna("<NULL>")
)

dups = df["_application_key"].duplicated(keep=False)
print("Rows involved in duplicate application keys:", int(dups.sum()))

## 8. Exclude rows where both title and summary are empty

In [ ]:
def blank_or_null(series):
    s = series.astype("string")
    return s.isna() | s.fillna("").str.strip().eq("")

no_text = blank_or_null(df["ApplicationTitle"]) & blank_or_null(df["ApplicationSummary"])

df_rejected = df.loc[no_text].copy()
df_rejected["rejection_reason"] = "NO_MODEL_TEXT"

df_primary_input = df.loc[~no_text].copy()

print("Total:", len(df))
print("Primary model input:", len(df_primary_input))
print("Rejected:", len(df_rejected))

if len(df_primary_input) == 0:
    raise ValueError("No usable model text.")

## 9. Run revised shared preprocessing

In [ ]:
df_cleaned = preprocess_text_fields(
    df=df_primary_input.copy(),
    text_fields=MODEL_TEXT_FIELDS,
    new_field_name="PROCESSED_TEXT",
    n_jobs=N_JOBS,
    batch_size=250,
)

if "PROCESSED_TEXT" not in df_cleaned.columns:
    raise ValueError("PROCESSED_TEXT was not created.")

display(df_cleaned[
    ["ApplicationID","ApplicationOriginSource",
     "ApplicationTitle","ApplicationSummary","PROCESSED_TEXT"]
].head(3))

## 10. Load and validate the single joblib

In [ ]:
model_dict = joblib.load(MAIN_MODEL_PATH)

if not isinstance(model_dict, dict):
    raise TypeError(f"Expected dict, found {type(model_dict)}")

required = {"vectorizer","models","thresholds","mlb"}
missing = required - set(model_dict.keys())
if missing:
    raise ValueError(f"Model artefact missing keys: {sorted(missing)}")

vectorizer = model_dict["vectorizer"]
models = model_dict["models"]
thresholds = np.asarray(model_dict["thresholds"]).reshape(-1)
mlb = model_dict["mlb"]

n_models = len(models)
n_thresholds = len(thresholds)
n_classes = len(mlb.classes_)

print("models:", n_models)
print("thresholds:", n_thresholds)
print("mlb classes:", n_classes)

if not (n_models == n_thresholds == n_classes):
    raise ValueError("Model package component lengths do not match.")

## 11. Vectorise and run primary inference

In [ ]:
X = vectorizer.transform(df_cleaned["PROCESSED_TEXT"])

probabilities = np.column_stack([
    model.predict_proba(X)[:, 1]
    for model in models
])

if probabilities.shape != (len(df_cleaned), n_classes):
    raise ValueError(f"Unexpected probability shape: {probabilities.shape}")

predictions = (probabilities >= thresholds.reshape(1,-1)).astype(int)

counts = predictions.sum(axis=1)
print("Zero categories:", int((counts == 0).sum()))
print("One category:", int((counts == 1).sum()))
print("Multiple categories:", int((counts > 1).sum()))

## 12. Decode primary predictions

In [ ]:
decoded_labels = [list(x) for x in mlb.inverse_transform(predictions)]
df_cleaned["_primary_pred_labels"] = decoded_labels
df_cleaned["_primary_prediction_count"] = [len(x) for x in decoded_labels]

display(df_cleaned[
    ["ApplicationID","ApplicationOriginSource",
     "_primary_pred_labels","_primary_prediction_count"]
].head(10))

## 13. Identify unresolved primary records

In [ ]:
resolved_mask = df_cleaned["_primary_prediction_count"] > 0

df_primary_resolved = df_cleaned.loc[resolved_mask].copy()
df_primary_unresolved = df_cleaned.loc[~resolved_mask].copy()
df_primary_unresolved["unresolved_reason"] = "NO_PRIMARY_CATEGORY_ABOVE_THRESHOLD"

print("Resolved:", len(df_primary_resolved))
print("Unresolved:", len(df_primary_unresolved))

# 14. FUTURE FALLBACK MODEL HOOK

Do not implement the second model yet.

Later this block should take only `df_primary_unresolved`, run the higher-level fallback model, and return rows with the same fields used downstream.

In [ ]:
# FUTURE FALLBACK MODEL PLACEHOLDER
df_fallback_predictions = pd.DataFrame(columns=[
    "ApplicationID",
    "ApplicationOriginSource",
    "category_id",
    "score",
    "prediction_source",
])

print("Fallback disabled.")
print("Rows reserved for future fallback:", len(df_primary_unresolved))

## 15. Convert primary multi-label predictions to long format

In [ ]:
rows = []

for row_pos, (_, source_row) in enumerate(df_cleaned.iterrows()):
    selected = np.flatnonzero(predictions[row_pos] == 1)
    for class_idx in selected:
        rows.append({
            "ApplicationID": source_row["ApplicationID"],
            "ApplicationOriginSource": source_row["ApplicationOriginSource"],
            "category_id": mlb.classes_[class_idx],
            "score": float(probabilities[row_pos, class_idx]),
            "prediction_source": "PRIMARY",
        })

df_primary_predictions = pd.DataFrame(rows, columns=[
    "ApplicationID","ApplicationOriginSource",
    "category_id","score","prediction_source"
])

display(df_primary_predictions.head(10))

## 16. Combine primary and future fallback outputs

In [ ]:
df_all_predictions = pd.concat(
    [df_primary_predictions, df_fallback_predictions],
    ignore_index=True
)
print("Combined rows:", len(df_all_predictions))

## 17. Build stakeholder output schema

In [ ]:
run_timestamp = datetime.now()
model_run_date = run_timestamp.date()

df_output = df_all_predictions.copy()
df_output["model_run_date"] = model_run_date
df_output["score_type"] = SCORE_TYPE
df_output["taxonomy"] = TAXONOMY_VALUE
df_output["score"] = pd.to_numeric(df_output["score"], errors="raise").round(2)

if len(df_output) > 0:
    category_numeric = pd.to_numeric(df_output["category_id"], errors="coerce")
    if category_numeric.isna().any():
        raise ValueError("Some category_id values are not numeric.")
    if not np.allclose(category_numeric, np.round(category_numeric)):
        raise ValueError("Some category_id values are not whole numbers.")
    df_output["category_id"] = np.round(category_numeric).astype("int64")
else:
    df_output["category_id"] = pd.Series(dtype="int64")

FINAL_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "model_run_date",
    "category_id",
    "score_type",
    "score",
    "taxonomy",
]

df_output = df_output[FINAL_COLUMNS]
display(df_output.head(10))

## 18. Validate final output

In [ ]:
OUTPUT_KEY = ["ApplicationID","ApplicationOriginSource","category_id"]

if df_output.duplicated(subset=OUTPUT_KEY, keep=False).any():
    raise ValueError("Duplicate output key detected.")

if len(df_output) > 0 and (((df_output["score"] < 0) | (df_output["score"] > 1)).any()):
    raise ValueError("Scores must be between 0 and 1.")

if not (df_output["score_type"] == SCORE_TYPE).all():
    raise ValueError("Unexpected score_type.")
if not (df_output["taxonomy"] == TAXONOMY_VALUE).all():
    raise ValueError("Unexpected taxonomy.")

print("Output validation passed.")

## 19. Generate output filename

In [ ]:
filename_timestamp = run_timestamp.strftime("%Y%m%d%H%M")
output_filename = (
    f"{filename_timestamp}_{TAXONOMY_FILE_TOKEN}_"
    f"{MODEL_NAME}_{MODEL_VERSION}{OUTPUT_EXTENSION}"
)
local_output_path = OUTPUT_DIR / output_filename

print(output_filename)

## 20. Save locally

In [ ]:
def save_dataframe(df_to_save, path):
    if path.suffix.lower() == ".csv":
        df_to_save.to_csv(path, index=False)
    elif path.suffix.lower() == ".parquet":
        df_to_save.to_parquet(path, index=False)
    else:
        raise ValueError(f"Unsupported output format: {path.suffix}")

save_dataframe(df_output, local_output_path)
print("Saved:", local_output_path)

## 21. Save rejected and unresolved traceability files

In [ ]:
rejected_path = REJECTED_DIR / f"{filename_timestamp}_{MODEL_NAME}_{MODEL_VERSION}_rejected_no_model_text.csv"
df_rejected[[
    "ApplicationID","ApplicationOriginSource",
    "ApplicationTitle","ApplicationSummary","rejection_reason"
]].to_csv(rejected_path, index=False)

unresolved_path = UNRESOLVED_DIR / f"{filename_timestamp}_{MODEL_NAME}_{MODEL_VERSION}_primary_unresolved.csv"
df_primary_unresolved[[
    "ApplicationID","ApplicationOriginSource",
    "ApplicationTitle","ApplicationSummary",
    "PROCESSED_TEXT","unresolved_reason"
]].to_csv(unresolved_path, index=False)

print("Rejected:", rejected_path)
print("Unresolved:", unresolved_path)

## 22. Upload validated output to S3

In [ ]:
s3_output_key = S3_OUTPUT_PREFIX.rstrip("/") + "/" + output_filename

s3.upload_file(str(local_output_path), S3_BUCKET, s3_output_key)
uploaded = s3.head_object(Bucket=S3_BUCKET, Key=s3_output_key)

print(f"s3://{S3_BUCKET}/{s3_output_key}")
print("Bytes:", uploaded["ContentLength"])

## 23. Run summary

In [ ]:
run_summary = {
    "run_timestamp": run_timestamp.isoformat(),
    "model_run_date": str(model_run_date),
    "input_s3_uri": f"s3://{S3_BUCKET}/{selected_input_key}",
    "input_rows": int(len(df)),
    "rejected_no_model_text": int(len(df_rejected)),
    "primary_input_rows": int(len(df_cleaned)),
    "primary_resolved_rows": int(len(df_primary_resolved)),
    "primary_unresolved_rows": int(len(df_primary_unresolved)),
    "primary_prediction_rows": int(len(df_primary_predictions)),
    "fallback_enabled": False,
    "final_output_rows": int(len(df_output)),
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_artifact": MAIN_MODEL_PATH.name,
    "taxonomy": TAXONOMY_VALUE,
    "score_type": SCORE_TYPE,
    "data_creator": "Automated Process",
    "output_s3_uri": f"s3://{S3_BUCKET}/{s3_output_key}",
    "status": "SUCCESS",
}

summary_path = LOG_DIR / f"{filename_timestamp}_{MODEL_NAME}_{MODEL_VERSION}_run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

print(json.dumps(run_summary, indent=2))

# Validation sequence before productionising

1. Run this notebook against the same test dataset used by the revised `inference.ipynb`.
2. Compare probability shapes, predicted label lists and unresolved counts.
3. Then run against the stakeholder sample file fetched from S3.
4. Confirm the final output matches the agreed schema and naming convention.
5. Only after that, extract notebook logic into `src/`, pin package versions, Dockerise, and schedule through Airflow.